# 🤖 Customer Churn Prediction AI Agent
### Google & Kaggle 5-Day AI Agents: Intensive Vibe Coding Course — Capstone

---

## 📌 Problem Statement
Telecom companies lose thousands of customers every month. They have no way to know *which* customer is about to leave *before* they actually leave. By the time the customer cancels, it's too late.

## ✅ Solution
We train an ML model on 7,000+ real customer records, then wrap it inside a **Google Gemini AI Agent** using function calling — so anyone can describe a customer in plain English and get a churn risk + retention recommendation instantly.

## 🏗️ Architecture
```
User (plain English) → Gemini Agent → predict_churn_risk() tool → ML Model → Risk + Recommendation
```

## Step 1: Install Dependencies

In [ ]:
!pip install google-genai joblib scikit-learn pandas matplotlib seaborn -q

## Step 2: Load and Clean the Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv')

print('Shape:', df.shape)
print('\nFirst 3 rows:')
df.head(3)

In [ ]:
# Fix TotalCharges — stored as text, convert to number
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop customerID — not useful for prediction
df = df.drop(columns=['customerID'])

print('Missing values after cleaning:', df.isnull().sum().sum())
print('Final shape:', df.shape)
df.to_csv('telco_clean.csv', index=False)
print('Saved telco_clean.csv')

## Step 3: Exploratory Data Analysis (EDA)
Finding what actually drives churn before building the model.

In [ ]:
overall_churn = (df['Churn'] == 'Yes').mean() * 100
print(f'Overall churn rate: {overall_churn:.1f}%')

def churn_rate_by(col):
    return df.groupby(col)['Churn'].apply(lambda x: (x=='Yes').mean()*100).sort_values(ascending=False)

print('\nChurn by Contract:')
print(churn_rate_by('Contract'))
print('\nChurn by Internet Service:')
print(churn_rate_by('InternetService'))
print('\nChurn by Payment Method:')
print(churn_rate_by('PaymentMethod'))
print('\nAverage tenure (months):')
print(df.groupby('Churn')['tenure'].mean())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

contract_rates = churn_rate_by('Contract')
contract_rates.plot(kind='bar', ax=axes[0], color='#0F6E56')
axes[0].set_title('Churn rate by Contract type')
axes[0].set_ylabel('Churn rate (%)')
axes[0].axhline(overall_churn, color='red', linestyle='--')
axes[0].tick_params(axis='x', rotation=30)

df[df['Churn']=='No']['tenure'].plot(kind='hist', bins=30, alpha=0.6, label='Stayed', ax=axes[1], color='#0F6E56')
df[df['Churn']=='Yes']['tenure'].plot(kind='hist', bins=30, alpha=0.6, label='Churned', ax=axes[1], color='#993C1D')
axes[1].set_title('Tenure: Stayed vs Churned')
axes[1].set_xlabel('Tenure (months)')
axes[1].legend()

internet_rates = churn_rate_by('InternetService')
internet_rates.plot(kind='bar', ax=axes[2], color='#534AB7')
axes[2].set_title('Churn rate by Internet Service')
axes[2].set_ylabel('Churn rate (%)')
axes[2].axhline(overall_churn, color='red', linestyle='--')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Key Churn Drivers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_charts.png', dpi=150)
plt.show()
print('Charts saved!')

## Step 4: Build and Evaluate the ML Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

# Prepare features and target
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['Churn'])
X = pd.get_dummies(X, drop_first=True)
feature_columns = X.columns.tolist()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.1%}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Stayed','Churned']))

In [ ]:
# Confusion matrix chart
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Greens')
ax.set_xticks([0,1]); ax.set_xticklabels(['Stayed','Churned'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Stayed','Churned'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (Accuracy: {acc:.1%})')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=14)
plt.tight_layout()
plt.savefig('model_results.png', dpi=150)
plt.show()

# Save model
joblib.dump({'model': model, 'scaler': scaler, 'feature_columns': feature_columns}, 'churn_model_bundle.pkl')
print('Model saved as churn_model_bundle.pkl')

## Step 5: Build the AI Agent (Gemini + Function Calling)

This is the core of the capstone. The agent uses our trained ML model as a **tool** — Gemini decides when to call it and writes the final recommendation in plain English.

**Agent pattern:**
- LLM (Gemini) = brain / reasoning
- predict_churn_risk() = tool / facts

The LLM never guesses the churn number — it always calls our real trained model.

In [ ]:
import os
from google import genai
from google.genai import types

# --- Set your API key before running this cell ---
# os.environ['GEMINI_API_KEY'] = 'paste_your_key_here'

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

# The prediction tool
bundle = joblib.load('churn_model_bundle.pkl')
model_ml = bundle['model']
scaler_ml = bundle['scaler']
feat_cols = bundle['feature_columns']

def predict_churn_risk(customer: dict) -> dict:
    row = pd.DataFrame([customer])
    row_encoded = pd.get_dummies(row)
    row_encoded = row_encoded.reindex(columns=feat_cols, fill_value=0)
    row_scaled = scaler_ml.transform(row_encoded)
    prob = model_ml.predict_proba(row_scaled)[0][1]
    risk = 'High' if prob >= 0.6 else ('Medium' if prob >= 0.3 else 'Low')
    factors = []
    if customer.get('Contract') == 'Month-to-month': factors.append('Month-to-month contract')
    if customer.get('InternetService') == 'Fiber optic': factors.append('Fiber optic service')
    if customer.get('PaymentMethod') == 'Electronic check': factors.append('Electronic check payment')
    if customer.get('tenure', 99) < 12: factors.append('New customer under 12 months')
    if customer.get('OnlineSecurity') == 'No': factors.append('No online security add-on')
    return {'churn_probability': round(float(prob), 3), 'risk_level': risk, 'top_risk_factors': factors}

print('Tool ready!')

In [ ]:
# Register tool with Gemini and run the agent
predict_churn_tool = types.FunctionDeclaration(
    name='predict_churn_risk',
    description='Predicts a telecom customer churn risk and returns key risk factors.',
    parameters={
        'type': 'object',
        'properties': {
            'tenure': {'type': 'integer'}, 'Contract': {'type': 'string'},
            'InternetService': {'type': 'string'}, 'PaymentMethod': {'type': 'string'},
            'MonthlyCharges': {'type': 'number'}, 'TotalCharges': {'type': 'number'},
            'OnlineSecurity': {'type': 'string'}, 'gender': {'type': 'string'},
            'SeniorCitizen': {'type': 'integer'}, 'Partner': {'type': 'string'},
            'Dependents': {'type': 'string'}, 'PhoneService': {'type': 'string'},
            'MultipleLines': {'type': 'string'}, 'OnlineBackup': {'type': 'string'},
            'DeviceProtection': {'type': 'string'}, 'TechSupport': {'type': 'string'},
            'StreamingTV': {'type': 'string'}, 'StreamingMovies': {'type': 'string'},
            'PaperlessBilling': {'type': 'string'}
        },
        'required': ['tenure', 'Contract', 'InternetService', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
    }
)

tools = types.Tool(function_declarations=[predict_churn_tool])
config = types.GenerateContentConfig(
    tools=[tools],
    system_instruction=(
        'You are a customer retention assistant. When a user describes a customer, '
        'call predict_churn_risk to get the real churn probability. NEVER guess the number. '
        'Then explain the result simply and give 2-3 concrete retention recommendations.'
    )
)

def ask_agent(message):
    contents = [types.Content(role='user', parts=[types.Part(text=message)])]
    response = client.models.generate_content(model='gemini-2.5-flash', contents=contents, config=config)
    candidate = response.candidates[0]
    fc = next((p.function_call for p in candidate.content.parts if p.function_call), None)
    if fc is None:
        return response.text
    args = {k: v for k, v in fc.args.items()}
    print(f'[Agent called tool with: {args}]')
    tool_result = predict_churn_risk(args)
    print(f'[Tool returned: {tool_result}]\n')
    contents.append(candidate.content)
    contents.append(types.Content(role='user', parts=[types.Part(
        function_response=types.FunctionResponse(name='predict_churn_risk', response=tool_result)
    )]))
    final = client.models.generate_content(model='gemini-2.5-flash', contents=contents, config=config)
    return final.text

print('Agent ready!')

### Test 1: High Risk Customer

In [ ]:
response = ask_agent(
    'Customer Riya: 2 months tenure, fiber optic internet, electronic check payment, '
    'month-to-month contract, no online security, pays $70/month, total charges $140.'
)
print(response)

### Test 2: Low Risk Customer

In [ ]:
response = ask_agent(
    'Customer Rahul: 48 months tenure, DSL internet, credit card autopay, '
    'two-year contract, has online security, pays $45/month, total charges $2160.'
)
print(response)

### Test 3: Medium Risk Customer

In [ ]:
response = ask_agent(
    'Customer Priya: 14 months tenure, fiber optic, bank transfer autopay, '
    'one-year contract, no online security, pays $65/month, total charges $910.'
)
print(response)

## 📋 Summary of Findings

| Factor | Churn Rate |
|---|---|
| Month-to-month contract | 42.7% |
| One year contract | 11.3% |
| Two year contract | 2.8% |
| Fiber optic internet | 41.9% |
| DSL internet | 19.0% |
| Electronic check payment | 45.3% |
| Auto payment | ~15% |

**Model accuracy: 80.7%**

## 💡 Business Recommendations
1. Offer discounts to convert month-to-month customers to annual contracts
2. Investigate fiber optic pricing and service quality
3. Incentivize autopay migration (reduces churn by 3x)
4. Focus retention efforts on customers in their first 12 months

---
*Built for Google & Kaggle 5-Day AI Agents: Intensive Vibe Coding Course*